In [1]:
# driver_drowsiness.py
import cv2, os, threading, numpy as np
from collections import deque
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.optimizers import Adam
from playsound import playsound

In [3]:
# 🔧 Paths and config
eye_data_dir = "C:/Users/helin/OneDrive/Documents/Desktop/doit/eyedataset/mrleyedataset/"        # folders: Close-Eyes, Open-Eyes
yawn_data_dir = "C:/Users/helin/OneDrive/Documents/Desktop/doit/yawndataset/"         # folders: Yawn, No_Yawn
alarm_path = "C:/Users/helin/OneDrive/Documents/Desktop/doit/alarm.wav/" 
img_size = 224
batch_size = 32
epochs = 5    # adjust as needed

In [5]:
def train_model(data_dir, model_save_path):
    # Image generator with preprocessing and validation split
    datagen = ImageDataGenerator(preprocessing_function=preprocess_input, validation_split=0.2)
    
    train_gen = datagen.flow_from_directory(data_dir,
                                            target_size=(img_size, img_size),
                                            batch_size=batch_size,
                                            class_mode='categorical',
                                            subset='training')
    
    val_gen = datagen.flow_from_directory(data_dir,
                                          target_size=(img_size, img_size),
                                          batch_size=batch_size,
                                          class_mode='categorical',
                                          subset='validation')

    # Load MobileNetV2 as base model
    base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(img_size, img_size, 3))
    base_model.trainable = False

    # Add custom head
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(128, activation='relu')(x)
    output = Dense(train_gen.num_classes, activation='softmax')(x)
    
    # Compile model
    model = Model(inputs=base_model.input, outputs=output)
    model.compile(optimizer=Adam(1e-4), loss='categorical_crossentropy', metrics=['accuracy'])

    # ✅ Use steps_per_epoch to avoid generator issues
    steps_per_epoch = train_gen.samples // batch_size
    validation_steps = val_gen.samples // batch_size

    model.fit(train_gen,
              validation_data=val_gen,
              epochs=epochs,
              steps_per_epoch=steps_per_epoch,
              validation_steps=validation_steps)

    model.save(model_save_path)
    print(f"✅ Saved model to {model_save_path}")

# Uncomment these lines to train both models
train_model(eye_data_dir, "eye_model.h5")
train_model(yawn_data_dir, "yawn_model.h5")

Found 67919 images belonging to 2 classes.
Found 16979 images belonging to 2 classes.


C:\Users\helin\anaconda3\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/5
2122/2122 ━━━━━━━━━━━━━━━━━━━━ 2308s 1s/step - accuracy: 0.9433 - loss: 0.1490 - val_accuracy: 0.9000 - val_loss: 0.2690
Epoch 2/5
   1/2122 ━━━━━━━━━━━━━━━━━━━━ 20:56 592ms/step - accuracy: 0.9062 - loss: 0.1470

C:\Users\helin\anaconda3\Lib\site-packages\keras\src\trainers\epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


2122/2122 ━━━━━━━━━━━━━━━━━━━━ 297s 140ms/step - accuracy: 0.9062 - loss: 0.1470 - val_accuracy: 0.8993 - val_loss: 0.2682
Epoch 3/5
2122/2122 ━━━━━━━━━━━━━━━━━━━━ 2186s 1s/step - accuracy: 0.9748 - loss: 0.0710 - val_accuracy: 0.8996 - val_loss: 0.3275
Epoch 4/5
2122/2122 ━━━━━━━━━━━━━━━━━━━━ 284s 134ms/step - accuracy: 0.9688 - loss: 0.1265 - val_accuracy: 0.8994 - val_loss: 0.3387
Epoch 5/5
2122/2122 ━━━━━━━━━━━━━━━━━━━━ 2102s 991ms/step - accuracy: 0.9800 - loss: 0.0552 - val_accuracy: 0.8917 - val_loss: 0.3046


✅ Saved model to eye_model.h5
Found 4096 images belonging to 2 classes.
Found 1023 images belonging to 2 classes.
Epoch 1/5
128/128 ━━━━━━━━━━━━━━━━━━━━ 167s 1s/step - accuracy: 0.8214 - loss: 0.3833 - val_accuracy: 0.9062 - val_loss: 0.2657
Epoch 2/5
128/128 ━━━━━━━━━━━━━━━━━━━━ 90s 699ms/step - accuracy: 0.9522 - loss: 0.1450 - val_accuracy: 0.9113 - val_loss: 0.2282
Epoch 3/5
128/128 ━━━━━━━━━━━━━━━━━━━━ 95s 745ms/step - accuracy: 0.9647 - loss: 0.1120 - val_accuracy: 0.9073 - val_loss: 0.2570
Epoch 4/5
128/128 ━━━━━━━━━━━━━━━━━━━━ 83s 648ms/step - accuracy: 0.9724 - loss: 0.0885 - val_accuracy: 0.9274 - val_loss: 0.1985
Epoch 5/5
128/128 ━━━━━━━━━━━━━━━━━━━━ 82s 643ms/step - accuracy: 0.9810 - loss: 0.0768 - val_accuracy: 0.9415 - val_loss: 0.1640


✅ Saved model to yawn_model.h5


In [17]:
import urllib.request
import bz2

# Download from official dlib site
url = "http://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2"
filename = "shape_predictor_68_face_landmarks.dat.bz2"

print("Downloading... This may take a few minutes (~100MB)")
urllib.request.urlretrieve(url, filename)

# Extract the .bz2 file
print("Extracting...")
with bz2.open(filename, "rb") as f_in, open("shape_predictor_68_face_landmarks.dat", "wb") as f_out:
    f_out.write(f_in.read())

print("Done! You should see shape_predictor_68_face_landmarks.dat in your folder.")


Downloading... This may take a few minutes (~100MB)
Extracting...
Done! You should see shape_predictor_68_face_landmarks.dat in your folder.


In [1]:
# driver_drowsiness_hybrid_fusion.py
import cv2, dlib, threading, time
import numpy as np
from keras.models import load_model
from scipy.spatial import distance as dist
import pygame

# ---------------------------
# CONFIG / FILE PATHS
# ---------------------------
EYE_MODEL_PATH  = "eye_model.h5"
YAWN_MODEL_PATH = "yawn_model.h5"
PREDICTOR_PATH  = "shape_predictor_68_face_landmarks.dat"
ALARM_PATH      = "alarm.wav"

# ---------------------------
# Load models & dlib predictor
# ---------------------------
eye_model  = load_model(EYE_MODEL_PATH)
yawn_model = load_model(YAWN_MODEL_PATH)

detector  = dlib.get_frontal_face_detector()
predictor = dlib.shape_predictor(PREDICTOR_PATH)

# ---------------------------
# Audio Alarm (pygame) - 2s
# ---------------------------
pygame.mixer.init()
_alarm_lock = threading.Lock()
_alarm_playing = False

def play_alarm_2s():
    global _alarm_playing
    with _alarm_lock:
        if _alarm_playing:
            return
        _alarm_playing = True
    try:
        pygame.mixer.music.load(ALARM_PATH)
        pygame.mixer.music.play()
        time.sleep(2.0)
        pygame.mixer.music.stop()
    except Exception as e:
        print("Alarm error:", e)
    finally:
        with _alarm_lock:
            _alarm_playing = False

# ---------------------------
# Helpers
# ---------------------------
def eye_aspect_ratio(eye):
    A = dist.euclidean(eye[1], eye[5])
    B = dist.euclidean(eye[2], eye[4])
    C = dist.euclidean(eye[0], eye[3])
    return (A + B) / (2.0 * C + 1e-6)

def mouth_aspect_ratio(mouth):
    A = dist.euclidean(mouth[2], mouth[10])  # 51,59
    B = dist.euclidean(mouth[4], mouth[8])   # 53,57
    C = dist.euclidean(mouth[0], mouth[6])   # 49,55
    return (A + B) / (2.0 * C + 1e-6)

def preprocess_for_cnn(bgr, size=(224,224)):
    try:
        img = cv2.resize(bgr, size, interpolation=cv2.INTER_AREA)
        img = img.astype(np.float32) / 255.0
        return np.expand_dims(img, axis=0)
    except:
        return None

# ---------------------------
# PARAMETERS
# ---------------------------
EAR_THRESH = 0.25
MAR_THRESH = 0.60
FRAME_CONSEC = 12   # consecutive frames before alarm
COOLDOWN_S = 5.0    # seconds

consec_eye = 0
consec_yawn = 0
last_alert_time = 0.0

# ---------------------------
# Video loop
# ---------------------------
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("Cannot open webcam")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    h, w = frame.shape[:2]
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    rects = detector(gray, 0)
    if len(rects) == 0:
        cv2.imshow("Hybrid Drowsiness Detection (Fusion)", frame)
        if cv2.waitKey(1) == 27:
            break
        continue

    rect = max(rects, key=lambda r: r.width() * r.height())
    shape = predictor(gray, rect)
    pts = np.array([[p.x, p.y] for p in shape.parts()], dtype=np.int32)

    leftEye  = pts[42:48]
    rightEye = pts[36:42]
    mouthPts = pts[48:68]

    ear = (eye_aspect_ratio(leftEye) + eye_aspect_ratio(rightEye)) / 2.0
    mar = mouth_aspect_ratio(mouthPts)

    cv2.putText(frame, f"EAR:{ear:.2f}", (w-180,30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0,255,0), 2)
    cv2.putText(frame, f"MAR:{mar:.2f}", (w-180,60),
                cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0,255,255), 2)

    # Face ROI
    x1, y1, x2, y2 = rect.left(), rect.top(), rect.right(), rect.bottom()
    x1, y1 = max(0,x1), max(0,y1)
    x2, y2 = min(w,x2), min(h,y2)
    face = frame[y1:y2, x1:x2].copy()

    fh, fw = y2-y1, x2-x1
    eye_roi   = face[int(fh*0.22):int(fh*0.52), int(fw*0.20):int(fw*0.80)]
    mouth_roi = face[int(fh*0.62):int(fh*0.95), int(fw*0.22):int(fw*0.78)]

    # CNN Eye Prediction (fix: 0=Open, 1=Closed)
    eye_label = 0
    xeye = preprocess_for_cnn(eye_roi)
    if xeye is not None:
        p = eye_model.predict(xeye, verbose=0)[0]
        eye_label = int(np.argmax(p))

    # CNN Mouth Prediction
    yawn_label = 0
    xmouth = preprocess_for_cnn(mouth_roi)
    if xmouth is not None:
        q = yawn_model.predict(xmouth, verbose=0)[0]
        yawn_label = int(np.argmax(q))

    # ---------- Fusion logic ----------
    # Mouth: CNN + MAR fusion
    if (yawn_label == 1) or (mar > MAR_THRESH):
        yawn_label = 1
    else:
        yawn_label = 0

    # Eye: CNN + EAR fusion
    if (eye_label == 1) or (ear < EAR_THRESH):
        eye_label = 1  # Closed
    else:
        eye_label = 0  # Open
    # ---------------------------------

    # Display
    cv2.putText(frame, f"Eye: {'Closed' if eye_label==1 else 'Open'}",
                (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,0), 2)
    cv2.putText(frame, f"Mouth: {'Yawn' if yawn_label==1 else 'No Yawn'}",
                (x1, y2+25), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,255), 2)

    # Counters
    consec_eye  = consec_eye+1 if eye_label==1 else 0
    consec_yawn = consec_yawn+1 if yawn_label==1 else 0

    # Trigger alarm
    if (consec_eye >= FRAME_CONSEC) or (consec_yawn >= FRAME_CONSEC):
        if time.time() - last_alert_time > COOLDOWN_S:
            threading.Thread(target=play_alarm_2s).start()
            last_alert_time = time.time()
        # <<< MOVED ALERT TO TOP-LEFT >>>
        cv2.putText(frame, "!!! DROWSINESS ALERT !!!", (10, 80),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0,0,255), 3)

    cv2.imshow("Hybrid Drowsiness Detection (Fusion)", frame)
    if cv2.waitKey(1) == 27:
        break

cap.release()
cv2.destroyAllWindows()


pygame 2.6.1 (SDL 2.28.4, Python 3.12.3)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [11]:
# evaluate_models_metrics.py
from keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# -------------------------------
# 🔧 Paths and config
# -------------------------------
eye_data_dir = "C:/Users/helin/OneDrive/Documents/Desktop/doit/eyedataset/mrleyedataset/"
yawn_data_dir = "C:/Users/helin/OneDrive/Documents/Desktop/doit/yawndataset/"
img_size = 224
batch_size = 32

# Load trained models
eye_model = load_model("eye_model.h5")
yawn_model = load_model("yawn_model.h5")

# -------------------------------
# 🔹 Prepare test/validation data
# -------------------------------
datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

eye_val = datagen.flow_from_directory(
    eye_data_dir,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

yawn_val = datagen.flow_from_directory(
    yawn_data_dir,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

# -------------------------------
# 🔹 Evaluate Eye Model
# -------------------------------
print("\n🔍 Evaluating Eye Model...")
eye_loss, eye_acc = eye_model.evaluate(eye_val)
print(f"✅ Eye Model Accuracy: {eye_acc * 100:.2f}%")

eye_preds = eye_model.predict(eye_val)
eye_y_true = eye_val.classes
eye_y_pred = np.argmax(eye_preds, axis=1)

print("\nClassification Report (Eye Model):")
print(classification_report(eye_y_true, eye_y_pred, target_names=list(eye_val.class_indices.keys())))

print("Confusion Matrix (Eye Model):")
print(confusion_matrix(eye_y_true, eye_y_pred))

# -------------------------------
# 🔹 Evaluate Yawn Model
# -------------------------------
print("\n🔍 Evaluating Yawn Model...")
yawn_loss, yawn_acc = yawn_model.evaluate(yawn_val)
print(f"✅ Yawn Model Accuracy: {yawn_acc * 100:.2f}%")

yawn_preds = yawn_model.predict(yawn_val)
yawn_y_true = yawn_val.classes
yawn_y_pred = np.argmax(yawn_preds, axis=1)

print("\nClassification Report (Yawn Model):")
print(classification_report(yawn_y_true, yawn_y_pred, target_names=list(yawn_val.class_indices.keys())))

print("Confusion Matrix (Yawn Model):")
print(confusion_matrix(yawn_y_true, yawn_y_pred))


Found 16979 images belonging to 2 classes.
Found 1023 images belonging to 2 classes.

🔍 Evaluating Eye Model...
531/531 ━━━━━━━━━━━━━━━━━━━━ 368s 690ms/step - accuracy: 0.8912 - loss: 0.4878
✅ Eye Model Accuracy: 68.58%
531/531 ━━━━━━━━━━━━━━━━━━━━ 218s 408ms/step

Classification Report (Eye Model):
              precision    recall  f1-score   support

  Close-Eyes       0.61      0.99      0.76      8389
   Open-Eyes       0.98      0.39      0.55      8590

    accuracy                           0.69     16979
   macro avg       0.80      0.69      0.66     16979
weighted avg       0.80      0.69      0.65     16979

Confusion Matrix (Eye Model):
[[8336   53]
 [5281 3309]]

🔍 Evaluating Yawn Model...


C:\Users\helin\anaconda3\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


32/32 ━━━━━━━━━━━━━━━━━━━━ 23s 662ms/step - accuracy: 0.9424 - loss: 0.1511
✅ Yawn Model Accuracy: 81.13%
32/32 ━━━━━━━━━━━━━━━━━━━━ 16s 464ms/step

Classification Report (Yawn Model):
              precision    recall  f1-score   support

     no yawn       0.73      1.00      0.84       518
        yawn       1.00      0.62      0.76       505

    accuracy                           0.81      1023
   macro avg       0.86      0.81      0.80      1023
weighted avg       0.86      0.81      0.80      1023

Confusion Matrix (Yawn Model):
[[517   1]
 [192 313]]
